# 12_two_stage_model.ipynb

**Sistema de Dos Etapas para el Cold-Start Problem**

- **Stage 1** (juegos conocidos, pre-2016): RS embeddings → TSCV R²≈0.84
- **Stage 2** (juegos nuevos, post-2016): contenido puro sin RS

## Objetivo
¿Puede Stage 2 (TF-IDF 100d + Steam 2d + RAWG 12d + dev_rep 1d = 115d) superar Modelo 04 (R²=0.065)?

In [1]:
import pandas as pd
import numpy as np
import json, ast, sys, os, warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

INTERACTIONS = "../Data/interactions.parquet"
ITEMMAP      = "../Data/item2idx.json"
STEAM_GAMES  = "../Data/steam_games.json"
RAWG_CSV     = "../Data/rawg_enriched.csv"
DEV_REP      = "../Data/developer_reputation.npy"
CUTOFF       = pd.to_datetime('2016-01-01')

with open(ITEMMAP, "r") as f:
    item2idx = {k: int(v) for k, v in json.load(f).items()}
N = len(item2idx)

df_inter  = pd.read_parquet(INTERACTIONS)
target_df = df_inter.groupby('item_idx').size().reset_index(name='total_reviews')
y         = target_df.set_index('item_idx').reindex(range(N), fill_value=0)['total_reviews'].values
dev_rep   = np.load(DEV_REP)

print(f"Items: {N} | Target range: {y.min()}-{y.max()} (mean={y.mean():.1f})")
print(f"Dev rep coverage: {(dev_rep > 0).sum()} / {N}")

Items: 3682 | Target range: 1-3759 (mean=16.1)
Dev rep coverage: 2217 / 3682


In [2]:
def parse_list_col(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return []
    if isinstance(x, list): return [str(t).strip() for t in x if t]
    try:
        lst = eval(x)
        return [str(t).strip() for t in lst if t] if isinstance(lst, list) else []
    except:
        return []

def _parse_price(p):
    try: return float(p)
    except: return 0.0

# Steam metadata
games = []
with open(STEAM_GAMES, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: games.append(ast.literal_eval(line))
        except: pass

df_games = pd.json_normalize(games).rename(columns={'id': 'item_id'})
df_games['item_idx'] = df_games['item_id'].map(item2idx)
df_games = df_games.dropna(subset=['item_idx'])
df_games['item_idx'] = df_games['item_idx'].astype(int)
df_games['release_date_parsed'] = pd.to_datetime(df_games['release_date'], errors='coerce')
df_games['tag_text']    = df_games['tags'].apply(parse_list_col).apply(' '.join)
df_games['genre_text']  = df_games['genres'].apply(parse_list_col).apply(' '.join)
df_games['content_text']= df_games['tag_text'] + ' ' + df_games['genre_text']
df_games['price_num']   = df_games['price'].apply(_parse_price)
df_games['ea_flag']     = df_games['early_access'].apply(lambda x: 1 if x else 0)

# TF-IDF on Steam content
games_w_content = df_games[df_games['content_text'].str.strip() != ''].copy()
tfidf = TfidfVectorizer(max_features=100, min_df=2, max_df=0.5, ngram_range=(1,1))
# Fit TF-IDF only on pre-2016 games to avoid leakage from test set
_pre2016_content = games_w_content[
    games_w_content['release_date_parsed'] < CUTOFF
]['content_text']
tfidf.fit(_pre2016_content)
tfidf_matrix = tfidf.transform(games_w_content['content_text'])
item_to_tfidf = {int(r['item_idx']): i for i, (_, r) in enumerate(games_w_content.iterrows())}
print(f"TF-IDF: {tfidf_matrix.shape}")

# RAWG features (12d)
df_rawg = pd.read_csv(RAWG_CSV).drop_duplicates(subset=['item_idx'], keep='last').set_index('item_idx')
for col in ['genres', 'platforms', 'developers', 'publishers', 'tags']:
    if col in df_rawg.columns:
        df_rawg[col] = df_rawg[col].apply(parse_list_col)

ESRB_ORDER = {'Everyone':1,'Everyone 10+':2,'Teen':3,'Mature':4,'Adults Only':5}

def rawg_vec(idx):
    if idx not in df_rawg.index: return [0.]*12
    r = df_rawg.loc[idx]
    return [
        1.0 if r.get('rawg_id') is not None else 0.0,
        float(r['rawg_rating'])       if pd.notna(r.get('rawg_rating'))       else -1.0,
        0.0 if pd.notna(r.get('rawg_rating'))       else 1.0,
        float(r['metacritic'])         if pd.notna(r.get('metacritic'))        else -1.0,
        0.0 if pd.notna(r.get('metacritic'))        else 1.0,
        np.log1p(float(r['playtime_avg_h']))     if pd.notna(r.get('playtime_avg_h'))     else 0.0,
        np.log1p(float(r['rawg_ratings_count'])) if pd.notna(r.get('rawg_ratings_count')) else 0.0,
        float(len(parse_list_col(r.get('platforms',[])))),
        float(len(parse_list_col(r.get('genres',[])))),
        float(len(parse_list_col(r.get('developers',[])))),
        float(len(parse_list_col(r.get('publishers',[])))),
        float(ESRB_ORDER.get(r.get('esrb_rating',''), 0)),
    ]

rawg_arr = np.array([rawg_vec(i) for i in range(N)], dtype=np.float32)
print(f"RAWG: {rawg_arr.shape} | match: {int(rawg_arr[:,0].sum())}/{N}")


TF-IDF: (3194, 100)


RAWG: (3682, 12) | match: 3195/3682


In [3]:
# Build Stage 2 matrix: TF-IDF(100) + Steam(2) + RAWG(12) + dev_rep(1) = 115d
stage2_rows, valid_idxs = [], []
for idx in range(N):
    if idx not in item_to_tfidf: continue
    g = games_w_content[games_w_content['item_idx'] == idx]
    if g.empty: continue
    g = g.iloc[0]
    vec = np.concatenate([
        tfidf_matrix[item_to_tfidf[idx]].toarray().flatten(),  # 100
        [g['price_num'], g['ea_flag']],                         # 2
        rawg_arr[idx],                                          # 12
        [dev_rep[idx]],                                         # 1
    ])
    stage2_rows.append(vec)
    valid_idxs.append(idx)

X_s2 = np.array(stage2_rows, dtype=np.float32)
y_s2 = target_df.set_index('item_idx').reindex(valid_idxs, fill_value=0)['total_reviews'].values

data_df = pd.DataFrame({'item_idx': valid_idxs}).merge(
    df_games[['item_idx','release_date_parsed']], on='item_idx', how='left'
)
s2_train = data_df['release_date_parsed'].notna() & (data_df['release_date_parsed'] < CUTOFF)
s2_test  = data_df['release_date_parsed'].notna() & (data_df['release_date_parsed'] >= CUTOFF)
s2_train, s2_test = s2_train.values, s2_test.values

print(f"X_s2: {X_s2.shape}  (TF-IDF 100 + Steam 2 + RAWG 12 + dev_rep 1)")
print(f"Train: {s2_train.sum()} | Test: {s2_test.sum()}")

# Impute RAWG -1 values with train median
RAWG_OFFSET = 102  # after tfidf(100) + steam(2)
for ci in [1, 3]:  # rawg_rating=1, metacritic=3 within RAWG block
    col = RAWG_OFFSET + ci
    train_vals = X_s2[s2_train, col]
    valid_vals = train_vals[train_vals != -1.0]
    if len(valid_vals):
        med = float(np.median(valid_vals))
        X_s2[X_s2[:, col] == -1.0, col] = med
print("RAWG imputation done")

X_s2: (3194, 115)  (TF-IDF 100 + Steam 2 + RAWG 12 + dev_rep 1)
Train: 2620 | Test: 486
RAWG imputation done


In [4]:
X_tr, y_tr = X_s2[s2_train], y_s2[s2_train]
X_te, y_te = X_s2[s2_test],  y_s2[s2_test]

# Sort by release date so Optuna val set is always the most recent 20%
_sort_order = np.argsort(data_df.loc[s2_train, 'release_date_parsed'].values)
X_tr = X_tr[_sort_order]
y_tr = y_tr[_sort_order]

# Optuna tuning
sp = max(10, int(len(X_tr)*0.8))
X_opt, X_val = X_tr[:sp], X_tr[sp:]
y_opt, y_val = y_tr[:sp], y_tr[sp:]

def objective(trial):
    p = dict(
        n_estimators=trial.suggest_int('n_estimators',100,800),
        learning_rate=trial.suggest_float('learning_rate',1e-3,0.3,log=True),
        max_depth=trial.suggest_int('max_depth',3,8),
        min_child_weight=trial.suggest_int('min_child_weight',1,10),
        subsample=trial.suggest_float('subsample',0.5,1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree',0.5,1.0),
        gamma=trial.suggest_float('gamma',0.0,2.0),
        random_state=42, tree_method='hist', verbosity=0,
    )
    m = XGBRegressor(**p)
    m.fit(X_opt, y_opt)
    return float(mean_squared_error(y_val, m.predict(X_val))**0.5)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)
best_params = {**study.best_params, 'random_state':42, 'tree_method':'hist', 'verbosity':0}
print(f"Best val RMSE: {study.best_value:.4f}")


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 48.9976:   0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 48.9976:   2%|▏         | 1/50 [00:00<00:25,  1.90it/s]

Best trial: 0. Best value: 48.9976:   2%|▏         | 1/50 [00:02<00:25,  1.90it/s]

Best trial: 0. Best value: 48.9976:   4%|▍         | 2/50 [00:02<00:58,  1.23s/it]

...

Best trial: 30. Best value: 14.9817:  98%|█████████▊| 49/50 [00:31<00:00,  1.99it/s]

Best trial: 49. Best value: 14.6666:  98%|█████████▊| 49/50 [00:31<00:00,  1.99it/s]

Best trial: 49. Best value: 14.6666: 100%|██████████| 50/50 [00:31<00:00,  2.08it/s]

Best trial: 49. Best value: 14.6666: 100%|██████████| 50/50 [00:31<00:00,  1.57it/s]

Best val RMSE: 14.6666


In [5]:
model_s2 = XGBRegressor(**best_params)
model_s2.fit(X_tr, y_tr)
pred_te = model_s2.predict(X_te)

r2_temporal   = r2_score(y_te, pred_te)
rmse_temporal = mean_squared_error(y_te, pred_te)**0.5
mae_temporal  = mean_absolute_error(y_te, pred_te)
mape_temporal = mean_absolute_percentage_error(y_te, pred_te)

print("="*70)
print("STAGE 2 — Content-Only Model (115 features)")
print("="*70)
print(f"R²  temporal:  {r2_temporal:.6f}")
print(f"RMSE temporal: {rmse_temporal:.2f}")
print(f"MAE  temporal: {mae_temporal:.2f}")
print()
print("COMPARACION:")
print(f"  Modelo 04 (Metadata, 6d):     R²=0.0653  RMSE=55.00")
print(f"  Stage 2 (Content, 115d):      R²={r2_temporal:.4f}  RMSE={rmse_temporal:.2f}")
delta = r2_temporal - 0.0653
print(f"  Delta R²: {delta:+.4f}  {'MEJORA' if delta > 0 else 'No mejora'}")
print("="*70)

# Feature importance
imp = model_s2.feature_importances_
RAWG_NAMES = ['rawg_found','rawg_rating','rawg_rating_miss','metacritic','metacritic_miss',
              'playtime_log','ratings_cnt_log','n_platforms','n_genres','n_developers','n_publishers','esrb']

imp_tfidf = imp[:100]; imp_steam = imp[100:102]
imp_rawg  = imp[102:114]; imp_dr = imp[114:]

print("\nFEATURE IMPORTANCE BY BLOCK:")
for blk_name, blk_imp, names in [
    ('TF-IDF (100d)', imp_tfidf, None),
    ('Steam (2d)',     imp_steam, ['price','early_access']),
    ('RAWG (12d)',     imp_rawg,  RAWG_NAMES),
    ('Dev rep (1d)',   imp_dr,    ['dev_rep']),
]:
    total = blk_imp.sum(); pct = total/imp.sum()*100
    print(f"  {blk_name:<20s}: {total:.4f} ({pct:.1f}%)")
    if names:
        top = sorted(zip(names, blk_imp), key=lambda x: x[1], reverse=True)[:4]
        for nm, v in top:
            if v > 0.001: print(f"      {nm}: {v:.6f}")

STAGE 2 — Content-Only Model (115 features)
R²  temporal:  0.073666
RMSE temporal: 54.76
MAE  temporal: 13.97

COMPARACION:
  Modelo 04 (Metadata, 6d):     R²=0.0653  RMSE=55.00
  Stage 2 (Content, 115d):      R²=0.0737  RMSE=54.76
  Delta R²: +0.0084  MEJORA

FEATURE IMPORTANCE BY BLOCK:
  TF-IDF (100d)       : 0.4786 (47.9%)
  Steam (2d)          : 0.0020 (0.2%)
      price: 0.002030
  RAWG (12d)          : 0.4305 (43.0%)
      playtime_log: 0.233930
      rawg_rating: 0.101357
      ratings_cnt_log: 0.072834
      n_platforms: 0.011844
  Dev rep (1d)        : 0.0889 (8.9%)
      dev_rep: 0.088879


In [6]:
# TSCV
TSCV_WINDOWS = [
    ('2013-07-01','2014-01-01'),('2014-01-01','2014-07-01'),
    ('2014-07-01','2015-01-01'),('2015-01-01','2015-07-01'),
    ('2015-07-01','2016-01-01'),
]
item_dates = data_df['release_date_parsed'].values
ts_results = []
print("="*70); print("TSCV — Stage 2"); print("="*70)
for tc_str, sc_str in TSCV_WINDOWS:
    tc, sc = pd.Timestamp(tc_str), pd.Timestamp(sc_str)
    t_msk = np.array([pd.notna(d) and d < tc for d in item_dates])
    e_msk = np.array([pd.notna(d) and d >= tc and d < sc for d in item_dates])
    if e_msk.sum() < 5: continue
    X_f = X_s2.copy()
    for ci in [1,3]:
        col = RAWG_OFFSET+ci
        tv = X_f[t_msk,col]; vv = tv[tv!=-1.]; 
        if len(vv): X_f[X_f[:,col]==-1., col] = float(np.median(vv))
    m = XGBRegressor(**best_params); m.fit(X_f[t_msk], y_s2[t_msk])
    p = m.predict(X_f[e_msk])
    r2 = r2_score(y_s2[e_msk],p); rmse = mean_squared_error(y_s2[e_msk],p)**0.5
    ts_results.append({'R2':r2,'RMSE':rmse})
    print(f"  {tc_str} -> {sc_str}: n={e_msk.sum():3d} | R2={r2:.4f} | RMSE={rmse:.2f}")

ts_df = pd.DataFrame(ts_results)
r2_tscv_mean,r2_tscv_std = ts_df['R2'].mean(),ts_df['R2'].std()
rmse_tscv_mean,rmse_tscv_std = ts_df['RMSE'].mean(),ts_df['RMSE'].std()
print(f"  -> Stage 2 TSCV R²={r2_tscv_mean:.4f} ± {r2_tscv_std:.4f}")
print(f"  -> Stage 1 TSCV R²=0.8410 (Modelo 08, RS+TF-IDF)")

# KFold
valid_s2 = data_df['release_date_parsed'].notna().values
X_kf,y_kf = X_s2[valid_s2],y_s2[valid_s2]
kf_res = []
for ti,ei in KFold(n_splits=5,shuffle=True,random_state=42).split(X_kf):
    m = XGBRegressor(**best_params); m.fit(X_kf[ti],y_kf[ti])
    p = m.predict(X_kf[ei])
    kf_res.append({'R2':r2_score(y_kf[ei],p),'RMSE':mean_squared_error(y_kf[ei],p)**0.5})
kf_df = pd.DataFrame(kf_res)
r2_kfold_mean,r2_kfold_std = kf_df['R2'].mean(),kf_df['R2'].std()
rmse_kfold_mean = kf_df['RMSE'].mean()
print(f"  -> KFold R²={r2_kfold_mean:.4f} ± {r2_kfold_std:.4f}")

TSCV — Stage 2


  2013-07-01 -> 2014-01-01: n=222 | R2=0.6178 | RMSE=48.74


  2014-01-01 -> 2014-07-01: n=280 | R2=0.4256 | RMSE=28.25


  2014-07-01 -> 2015-01-01: n=299 | R2=0.5008 | RMSE=12.69


  2015-01-01 -> 2015-07-01: n=342 | R2=0.2742 | RMSE=22.60


  2015-07-01 -> 2016-01-01: n=367 | R2=0.5246 | RMSE=19.57
  -> Stage 2 TSCV R²=0.4686 ± 0.1285
  -> Stage 1 TSCV R²=0.8410 (Modelo 08, RS+TF-IDF)


  -> KFold R²=0.2580 ± 0.2830


In [7]:
sys.path.insert(0, os.path.abspath('.'))
from results_tracker import save_result, print_leaderboard

save_result(
    model_id='12',
    model_name='Two-Stage: Content Model',
    features='TF-IDF (100d) + Steam (2d) + RAWG (12d) + dev_rep (1d)',
    embeddings='none',
    metrics={
        'r2_temporal':   r2_temporal,  'rmse_temporal': rmse_temporal,
        'mae_temporal':  mae_temporal, 'mape_temporal': mape_temporal,
        'r2_tscv':       r2_tscv_mean, 'r2_tscv_std':  r2_tscv_std,
        'rmse_tscv':     rmse_tscv_mean,'rmse_tscv_std':rmse_tscv_std,
        'r2_kfold':      r2_kfold_mean, 'r2_kfold_std': r2_kfold_std,
        'rmse_kfold':    rmse_kfold_mean,
    },
    notes='Stage 2 del sistema two-stage: contenido puro para juegos post-2016',
)

print_leaderboard()

print("\n" + "="*70)
print("RESUMEN SISTEMA DOS ETAPAS")
print("="*70)
print("Stage 1 — juegos con historial (pre-2016):")
print("  Modelo 08 RS+TF-IDF+Numeric    TSCV R²=0.8410")
print()
print("Stage 2 — juegos nuevos (post-2016):")
print(f"  Modelo 04 Metadata (6d)        temporal R²=0.0653")
print(f"  Este modelo  Content (115d)    temporal R²={r2_temporal:.4f}")
delta = r2_temporal - 0.0653
print(f"  Delta: {delta:+.4f}  {'MEJORA sobre baseline' if delta>0 else 'No mejora baseline'}")
print("="*70)

 ID  Modelo                         R2 test     RMSE     MAE   MAPE%   R2 TSCV  R2 KFold
[13c]  Stage2 Content (log target)     0.2539     0.84    9.88     1.3    0.7570    0.7241
[12]  Two-Stage: Content Model        0.0737    54.76   13.97     4.7    0.4686    0.2580
[04]  Metadata Only                   0.0572    55.24   18.84     8.4   -0.2144    0.0710
[06]  Review Text Emb                -0.0107    57.20   18.76     8.3   -0.1720    0.0072
[05]  RS + Metadata                  -0.0161    57.35    9.16     0.6    0.8888    0.0110
[11b]  Hybrid + Dev Reputation        -0.0163    57.36    9.40     0.8    0.8580    0.3073
[10]  RS + Reviews + RAWG            -0.0163    57.36    9.14    46.6    0.8634    0.4330
[03]  RS Embeddings Only             -0.0184    57.42    9.64     1.1    0.8861    0.4510
[09]  RS + Review Text               -0.0213    57.50    9.27    49.1    0.8733    0.3359
[08]  Hybrid Collab-Content          -0.0240    57.57    9.37     0.4    0.8562    0.2945
[11a]  RS